# Solving the OCP via indirect approach, SODE low thrust

first order eqs:
$$
\dot{x} = v_x, \dot{y}
$$


In [ ]:
import sympy
import numpy as np
import function_definitions as field_funcs
import helper_funcs as hfct
import pickle
import scipy
import scipy.optimize as opt
import OCP_construction_functions as OCP_funcs
import copy

In [ ]:
alpha_choice = 1
gamma_choice = 0.5
iterator = 0 # 0-11
N_choice= [50,100,150,200,250,300,350,400,450,500,550,600][iterator]

OCP_parameters = {
        "T": 28.,   #s
        "N":N_choice,     #240, 160, 80, 40, 20
        "q0": [sympy.Matrix([4.,0.])],        #(m,1)
        "dq0": [sympy.Matrix([0.,4.])],       #(m/s,1/s)
        "qT": [sympy.Matrix([5.,0.])],       #(r,phi) #mayer term does not consider phiT, rather automatically uses the one of the cartesian case
        "dqT": [sympy.Matrix([0.,0])],       #(vr,vphi)
        "alpha": alpha_choice,
        "gamma": gamma_choice,
        'folder_name': 'n_low_thrust/polar_variables',
        'RK_b': sympy.Matrix([1]),
        'RK_c': sympy.Matrix([1/2]),
        'RK_a': sympy.Matrix([[1/2]]),
        #specific to model
        "m":[1,1.,1.,1.,1.,1.],  # kg
        "M":10,
        "G": 1., #m/s**2
        "Aq": 1.0, #  kg/s^2
        "Adq": 50.0, # kg/s,
        'distance_between_satellites': np.pi/3,
        "base_variable_names": ["r", "phi"],
        "variable_names": []
    }
OCP_parameters['n_satellites'] = len(OCP_parameters["m"])
OCP_parameters["base_variable_dim_q"] = len(OCP_parameters["base_variable_names"])
OCP_parameters["dim_q"] = len(OCP_parameters["variable_names"])
OCP_parameters["dim_u"] = OCP_parameters['n_satellites']
OCP_parameters["dqT"][0][1] =sympy.sqrt(OCP_parameters["G"]*OCP_parameters["M"]/OCP_parameters["qT"][0][0]**3)
OCP_parameters["dq0"][0][1] =sympy.sqrt(OCP_parameters["G"]*OCP_parameters["M"]/OCP_parameters["q0"][0][0]**3)

single_skip = True
for i in range(OCP_parameters["n_satellites"]):
    for tmp in OCP_parameters["base_variable_names"]:
        OCP_parameters["variable_names"].append(tmp+str(i+1))
    if single_skip:
        single_skip=False
        continue
    OCP_parameters["q0"].append(OCP_parameters["q0"][-1])
    OCP_parameters["dq0"].append(OCP_parameters["dq0"][-1])
    OCP_parameters["dqT"].append(OCP_parameters["dqT"][-1])
    OCP_parameters["qT"].append(OCP_parameters["qT"][-1])


OCP_parameters["q0"]=sympy.Matrix(sympy.flatten(OCP_parameters["q0"]))
OCP_parameters["qT"]=sympy.Matrix(sympy.flatten(OCP_parameters["qT"]))
OCP_parameters["dqT"]=sympy.Matrix(sympy.flatten(OCP_parameters["dqT"]))
OCP_parameters["dq0"]=sympy.Matrix(sympy.flatten(OCP_parameters["dq0"]))

OCP_parameters["h"] = OCP_parameters["T"]*1.0/N_choice
OCP_parameters["times"] = sympy.Matrix([i*OCP_parameters["h"] for i in range(OCP_parameters["N"]+1)])


In [ ]:
def f_vec_n_satellites_polar(q,v,params):
    n_satellites = params["n_satellites"]
    dimq_base =  params["base_variable_dim_q"]
    f_vec = []
    for n in range(n_satellites):
        q_k = q[n*dimq_base: (n+1)*dimq_base]
        v_k = v[n*dimq_base: (n+1)*dimq_base]
        f_vec.append(field_funcs.f_vec_polar(q_k,v_k,params))

    f_vec = sympy.Matrix(sympy.flatten(f_vec))
    return f_vec

def rho_vec_n_polar(q,v,params):
    n_satellites = params["n_satellites"]
    dimq_base =  params["base_variable_dim_q"]
    dim_q = params["dim_q"]
    rho_vec = sympy.zeros(dim_q,n_satellites)
    
    for n in range(n_satellites):
        q_k = q[n*dimq_base: (n+1)*dimq_base]
        v_k = v[n*dimq_base: (n+1)*dimq_base]
        rho_vec[n*dimq_base:(n+1)*dimq_base,n] = field_funcs.rho_vec_polar(q_k,v_k,params)

    return rho_vec


def g_mat_n_polar(q,params):
    n_satellites = params["n_satellites"]
    return sympy.eye(n_satellites)


def mayer_term_n_polar(q,v,params):
    n_satellites = params["n_satellites"]
    dimq_base =  params["base_variable_dim_q"]
    mayer_vec = 0
    rvec = q[::2]
    phivec = q[1::2]
    vrvec = v[::2]
    vphivec = v[1::2]
    rvecT = params["qT"][::2]
    phivecT = params["qT"][1::2]
    vrvecT = params["dqT"][::2]
    vphivecT = params["dqT"][1::2]
    angle_distance = params["distance_between_satellites"]
    for n in range(n_satellites-1):
        phik1 = phivec[n+1]
        phik = phivec[n]
        q_term = params["Aq"]* ( (phik1-phik -angle_distance)**2) 
        mayer_vec += q_term

    for n in range(n_satellites):
        q_term += params["Aq"]* ((rvec[n]-rvecT[n])**2 ) 


        dq_term = params["Adq"]* (vrvec[n]**2 + (vphivec[n]-vphivecT[n])**2)
        mayer_vec+=q_term+ dq_term

    return sympy.Matrix([mayer_vec])

# Function definition

In [ ]:
continuous_eq = OCP_funcs.Direct_continuous_generator(OCP_parameters,f_func=f_vec_n_satellites_polar,rho_func=rho_vec_n_polar,running_cost_func=field_funcs.running_cost_polar,mayer_func=mayer_term_n_polar,g_func=g_mat_n_polar)


# initial guess creation

In [ ]:
# save_data_file = 'data/low_thrust/polar_variables/data_a=1g=0.5/data_a=1g=0.5N=500.pkl'


# OCP_parameters_np_version = hfct.sympy_to_np_dict(OCP_parameters,alpha_choice,gamma_choice)
# parameters=OCP_parameters_np_version

# with open(save_data_file, 'rb') as files:
#     initial_guess_data = pickle.load(files)

# q_d_start_guess = np.array(initial_guess_data['q_d_new'])
# lam_d_start_guess =np.array(initial_guess_data['lambda_d_new'])
# u_d_start_guess = initial_guess_data['u_d_new']

# cs_q =  scipy.interpolate.CubicSpline(np.linspace(0,parameters["T"],len(q_d_start_guess)), q_d_start_guess.reshape([len(q_d_start_guess),2]))
# cs_lam = scipy.interpolate.CubicSpline(np.linspace(0,parameters["T"],len(lam_d_start_guess)), lam_d_start_guess.reshape([len(lam_d_start_guess),2]))  
# cs_u= scipy.interpolate.CubicSpline(np.linspace(0,parameters["T"],len(u_d_start_guess)),u_d_start_guess)

# U_d_1_base = cs_u(np.array(parameters["times"]) + parameters["gamma"]*parameters["h"]).reshape(len(parameters["times"]),1,1)
# U_d_2_base = cs_u(np.array(parameters["times"]) + (1-parameters["gamma"])*parameters["h"]).reshape(len(parameters["times"]),1,1)

# q_d_base = cs_q(parameters["times"]).reshape(len(parameters["times"]),2,1)
# lambda_d_base = cs_lam(parameters["times"]).reshape(len(parameters["times"]),2,1)
# mu_base = np.array([[0.1,0.1]])
# nu_base = np.array([[0.1,0.1]])
# v_q_use =[]

# U_d_1_use,U_d_2_use = [],[]
# q_d_use,lambda_d_use = [],[]
# mu_use,nu_use = [],[]
# for i in range(N_choice +1):
#     U_d_1_use.append(U_d_1_base[i])
#     U_d_2_use.append(U_d_2_base[i])
#     q_d_use.append(q_d_base[i])
#     lambda_d_use.append(lambda_d_base[i])
#     for n in range(OCP_parameters["n_satellites"]-1):
#         U_d_1_use[-1] = np.append(U_d_1_use[-1],U_d_1_base[i])
#         U_d_2_use[-1]= np.append(U_d_2_use[-1],U_d_2_base[i])
#         q_d_use[-1] = np.append(q_d_use[-1],q_d_base[i]) 
#         lambda_d_use[-1] = np.append(lambda_d_use[-1],lambda_d_base[i]) 
   
# for n in range(OCP_parameters["n_satellites"]):
#     mu_use.append(mu_base)
#     nu_use.append(nu_base)

# U_d_1_use = np.array(U_d_1_use).flatten().reshape([(N_choice+1),OCP_parameters["dim_u"],1])
# U_d_2_use = np.array(U_d_2_use).flatten().reshape([(N_choice+1),OCP_parameters["dim_u"],1])
# q_d_use = np.array(q_d_use).flatten().reshape([(N_choice+1),OCP_parameters["dim_q"],1])
# lambda_d_use = np.array(lambda_d_use).flatten().reshape([(N_choice+1),OCP_parameters["dim_q"],1])
# mu_use = np.array(mu_use).flatten().reshape([1,OCP_parameters['dim_q']])
# nu_use = np.array(nu_use).flatten().reshape([1,OCP_parameters['dim_q']])


In [ ]:
#initial guess in polar coordinates trafo

def polar_to_cartesian(q_polar):
    r,phi = q_polar
    q_cartesian = np.array([r * np.cos(phi), r * np.sin(phi)])
    return q_cartesian


def cartesian_to_polar(q_cartesian):
    x,y = q_cartesian
    r = np.sqrt(x**2 + y**2)
    q_polar = np.array([r, 0])
    if x>0 and y>=0:
        q_polar[1] = np.arccos(x/r)
    elif x<0 and y>=0:
        q_polar[1] =  np.arccos(x/r)
    elif  x<0 and y < 0:
        q_polar[1] = 2*np.pi- np.arccos(x/r)
    else:
        q_polar[1] = 2*np.pi  - np.arccos(x/r)
    return q_polar 


# initial guess from working example

In [ ]:
save_data_file = 'data/n_low_thrust/polar_variables/data_a=1g=0.5/data_a=1g=0.5N=50.pkl'


OCP_parameters_np_version = hfct.sympy_to_np_dict(OCP_parameters,alpha_choice,gamma_choice)

with open(save_data_file, 'rb') as files:
    initial_guess_data = pickle.load(files)

OCP_parameters_np_version = hfct.sympy_to_np_dict(OCP_parameters,alpha_choice,gamma_choice)
parameters = OCP_parameters_np_version

q_d_start_guess = np.array(initial_guess_data['q_d_new'])
lam_d_start_guess =np.array(initial_guess_data['lambda_d_new'])
u_d_start_guess = initial_guess_data['u_d_new']
v_y_d_start_guess = np.array(initial_guess_data['v_y_d_new'])
cs_q =  scipy.interpolate.CubicSpline(np.linspace(0,parameters["T"],len(q_d_start_guess)), q_d_start_guess)
cs_vy =  scipy.interpolate.CubicSpline(np.linspace(0,parameters["T"],len(v_y_d_start_guess)), v_y_d_start_guess)
cs_lam = scipy.interpolate.CubicSpline(np.linspace(0,parameters["T"],len(lam_d_start_guess)), lam_d_start_guess)  
cs_u= scipy.interpolate.CubicSpline(np.linspace(0,parameters["T"],len(u_d_start_guess)),u_d_start_guess)

U_d_1_use = cs_u(np.array(parameters["times"]) + parameters["gamma"]*parameters["h"])
U_d_2_use = cs_u(np.array(parameters["times"]) + (1-parameters["gamma"])*parameters["h"])

q_d_use = cs_q(parameters["times"])
lambda_d_use = cs_lam(parameters["times"])
mu_base = np.array([[0.1,0.1]])
nu_base = np.array([[0.1,0.1]])

mu_use,nu_use = [],[]

for n in range(parameters["n_satellites"]): 
    mu_use.append(mu_base)
    nu_use.append(nu_base)

U_d_1_use = np.array(U_d_1_use)
U_d_2_use = np.array(U_d_2_use)
v_y_d_use = cs_vy(parameters['times'])
q_d_use = np.array(q_d_use)
lambda_d_use = np.array(lambda_d_use)
mu_use = np.array(mu_use).flatten().reshape([1,parameters['dim_q']])
nu_use = np.array(nu_use).flatten().reshape([1,parameters['dim_q']])


# Evolution via state eq

In [ ]:
discrete_equations = OCP_funcs.discrete_standard_direct_eq_generator(continuous_eq)

# lambdify the equations for root finding

In [ ]:
#KKT direct approach
standard_direct_midpoint_KKT = discrete_equations.calc_KKT()
lambdified_KKT =  sympy.lambdify(standard_direct_midpoint_KKT[1],standard_direct_midpoint_KKT[0].subs(discrete_equations.h,OCP_parameters_np_version["h"]).subs(discrete_equations.cont_equations.parameters["alpha"],alpha_choice).subs(discrete_equations.cont_equations.parameters["gamma"],gamma_choice)) 
lambdified_KKT_eval = lambda x :lambdified_KKT(*x)


In [ ]:
#KKT direct approach with new Lagrangian
standard_direct_KKT_new = discrete_equations.calc_KKT_new()

lambdified_KKT_new =  sympy.lambdify(standard_direct_KKT_new[1],standard_direct_KKT_new[0].subs(discrete_equations.h,OCP_parameters_np_version["h"]).subs(discrete_equations.cont_equations.parameters["alpha"],alpha_choice).subs(discrete_equations.cont_equations.parameters["gamma"],gamma_choice))

lambdified_KKT_new_eval = lambda x :lambdified_KKT_new(*x)


In [ ]:
standard_direct_KKT_new_no_u = discrete_equations.calc_KKT_new_no_u()
lambdified_KKT_new_no_u =  sympy.lambdify(standard_direct_KKT_new_no_u[1],standard_direct_KKT_new_no_u[0].subs(discrete_equations.h,OCP_parameters_np_version["h"]).subs(discrete_equations.cont_equations.parameters["alpha"],alpha_choice).subs(discrete_equations.cont_equations.parameters["gamma"],gamma_choice))
lambdified_KKT_new_eval_no_u = lambda x :lambdified_KKT_new_no_u(*x)


In [ ]:

v_y_vec = continuous_eq.vy
p_y_vec = continuous_eq.py
p_y_legendre=discrete_equations.cont_equations.p_y()
v_sol = sympy.solve(sympy.Eq(p_y_legendre,p_y_vec),v_y_vec)
v_vec = sympy.Matrix([v_sol[tmp] for tmp in v_y_vec])

v_eval = lambda q,lam,p_y: v_vec.evalf(subs={"r":q[0],"phi":q[1],"lambda_r":lam[0],"lambda_phi":lam[1], "p_r":p_y[0],"p_phi":p_y[1],"p_lambda_r":p_y[2],"p_lambda_phi":p_y[3]})


In [ ]:



initial_guess =  list(np.concatenate([mu_use.flatten(),nu_use.flatten()])) #mu and nu
initial_guess_new = list(np.concatenate([mu_use.flatten(),nu_use.flatten()]))
initial_guess_new_no_u=[]
for tmp in q_d_use:
    initial_guess += list(tmp)
    initial_guess_new+=list(tmp.flatten())
    #vq guess
for tmp in v_y_d_use[:,:2]:
    initial_guess+= list(tmp.flatten())
#lambda terms
for tmp in v_y_d_use[:,2:]:
    initial_guess+= list(tmp.flatten())
for tmp in lambda_d_use:
    initial_guess += list(tmp.flatten())
    initial_guess_new += list(tmp.flatten())

initial_guess_new_no_u =copy.deepcopy( initial_guess_new)

initial_guess_new += list(U_d_1_use.flatten()) 
initial_guess_new += list(U_d_2_use.flatten()) 
initial_guess += list(U_d_1_use.flatten())
initial_guess += list(U_d_2_use.flatten())


In [ ]:
# import scipy.optimize as opt
# import time
# starttime = time.time()

# #standard direct approach
# standard_result_polar_KKT = opt.root(lambdified_KKT_eval,x0=initial_guess,method="lm")
# endtime = time.time()

# Dt_standard = endtime-starttime
# dt_standard = Dt_standard/standard_result_polar_KKT.nfev

In [ ]:
#new lagrangian approach
import time
starttime = time.time()

standard_result_polar_KKT_new = opt.root(lambdified_KKT_new_eval,x0=initial_guess_new,method="lm")
endtime = time.time()

Dt_new = endtime-starttime
dt_new = Dt_new/standard_result_polar_KKT_new.nfev

In [ ]:
starttime = time.time()

standard_result_polar_KKT_new_no_u = opt.root(lambdified_KKT_new_eval_no_u,x0=initial_guess_new_no_u,method="lm")
endtime = time.time()

Dt_new_no_u = endtime-starttime
dt_new_no_u = Dt_new_no_u/standard_result_polar_KKT_new_no_u.nfev

In [ ]:
standard_result_polar_KKT_new

In [ ]:
standard_result_polar_KKT_new_no_u

In [ ]:
print(dt_new)
print(dt_new_no_u)

In [ ]:
reshape_params = discrete_equations.cont_equations.parameters
reshape_N = reshape_params["N"]
reshape_dim_q = reshape_params["dim_q"]
reshape_dim_u = reshape_params["dim_u"]
mu_KKT_new,nu_KKT_new = standard_result_polar_KKT_new.x[:2*reshape_dim_q].reshape(2,reshape_dim_q)
q_d_KKT_new = standard_result_polar_KKT_new.x[2*reshape_dim_q:2*reshape_dim_q + reshape_dim_q * (reshape_N+1)].reshape(reshape_N + 1,reshape_dim_q)
lam_d_KKT_new = standard_result_polar_KKT_new.x[2*reshape_dim_q + reshape_dim_q * (reshape_N+1):2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1)].reshape(reshape_N + 1,reshape_dim_q)

U1_d_KKT_new = standard_result_polar_KKT_new.x[2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1):2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1) + (reshape_N+1)*reshape_dim_u].reshape(reshape_N + 1,reshape_params["dim_u"])
U2_d_KKT_new = standard_result_polar_KKT_new.x[2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1) + (reshape_N+1)*reshape_dim_u:].reshape(reshape_N + 1,reshape_params["dim_u"])

p_y_d_new = np.array([sympy.flatten(tmp) for tmp in discrete_equations.p_v_d_from_y_d(q_d_KKT_new,lam_d_KKT_new,U1_d_KKT_new,U2_d_KKT_new,OCP_parameters_np_version)])



u_vec_KKT_new = []
for i in range(len(q_d_KKT_new)):
    u_vec_KKT_new.append(discrete_equations.cont_equations.u_eval_from_new(q_d_KKT_new[i],lam_d_KKT_new[i],p_y_d_new[i][:2],p_y_d_new[i][2:],OCP_parameters))

u_vec_KKT_new = np.array(u_vec_KKT_new)

H_control_KKT_new = []
for i in range(len(q_d_KKT_new)):
    H_control_KKT_new.append(discrete_equations.cont_equations.new_control_H_eval(q_d_KKT_new[i],lam_d_KKT_new[i],p_y_d_new[i][:2],p_y_d_new[i][2:],u_vec_KKT_new[i],OCP_parameters_np_version))
H_control_KKT_new = np.array(H_control_KKT_new)  

In [ ]:
plt.plot(u_vec_KKT_new[:,0],label='new ud')
plt.plot(u_vec_KKT_new[:,1],'--',label='new other')
plt.plot(u_vec_KKT_new[:,2],'--',label='new other2')
plt.plot(u_vec_KKT_new[:,3],'--',label='new other3')
plt.legend()


In [ ]:
plt.plot(q_d_KKT_new[:,0])
plt.plot(q_d_KKT_new[:,2])
plt.plot(q_d_KKT_new[:,4])
plt.plot(q_d_KKT_new[:,6])
plt.plot(q_d_KKT_new[:,8])
plt.plot(q_d_KKT_new[:,10])

In [ ]:
plt.plot(q_d_KKT_new[:,1]/np.pi/2)
plt.plot(q_d_KKT_new[:,3]/np.pi/2)
plt.plot(q_d_KKT_new[:,5]/np.pi/2)
plt.plot(q_d_KKT_new[:,7]/np.pi/2)
plt.plot(q_d_KKT_new[:,9]/np.pi/2)
plt.plot(q_d_KKT_new[:,11]/np.pi/2)

In [ ]:
v_y_vec = continuous_eq.vy
p_y_vec = continuous_eq.py
q_subs_vec = continuous_eq.q
lam_subs_vec = continuous_eq.lamq

p_y_legendre=discrete_equations.cont_equations.p_y()
v_sol = sympy.solve(sympy.Eq(p_y_legendre,p_y_vec),v_y_vec)
v_vec = sympy.Matrix([v_sol[tmp] for tmp in v_y_vec])

v_eval = lambda q,lam,p_y: v_vec.subs([[tmp1,tmp2] for tmp1,tmp2 in zip(q_subs_vec, q)]).subs([[tmp1,tmp2] for tmp1,tmp2 in zip(lam_subs_vec, lam)]).subs([[tmp1,tmp2] for tmp1,tmp2 in zip(p_y_vec, p_y)])

v_y_d_new = []
for tmp1,tmp2,tmp3 in zip(q_d_KKT_new,lam_d_KKT_new,p_y_d_new):
    v_y_d_new.append(sympy.flatten(v_eval(tmp1,tmp2,tmp3)))
v_y_d_new = np.array(v_y_d_new) 



def conserved_quantity_cartesian(q,lam,vq,vlam):
    x,y = np.array(q)
    lamx,lamy = np.array(lam)
    vx,vy = np.array(vq)
    vlamx,vlamy = np.array(vlam)

    return x*vlamy - y *vlamx -vx*lamy + vy*lamx  

I_new_evo = []
I_new_evo = p_y_d_new.T[1:12:2]


I_new_evo= np.array(I_new_evo,float)

    

In [ ]:
plt.plot(OCP_parameters_np_version["times"],v_y_d_new.transpose()[1],'--',label='new approach lam_r')
plt.plot(OCP_parameters_np_version["times"],v_y_d_new.transpose()[3],'--',label='new approach lam_r')
plt.plot(OCP_parameters_np_version["times"],v_y_d_new.transpose()[5],'--',label='new approach lam_r')
plt.plot(OCP_parameters_np_version["times"],v_y_d_new.transpose()[5],'--',label='new approach lam_r')
plt.plot(OCP_parameters_np_version["times"],v_y_d_new.transpose()[9],'--',label='new approach lam_r')
plt.plot(OCP_parameters_np_version["times"],v_y_d_new.transpose()[11],'--',label='new approach lam_r')


# plot results

In [ ]:

q_d_new_KKT_sol_cart = []
import sys
from pathlib import Path 
foldername = 'a' + str(alpha_choice) + 'g' + str(gamma_choice)


for val in q_d_KKT_new:
    step_data = []
    for i in range(OCP_parameters["n_satellites"]):
        step_data.append(polar_to_cartesian(val[i*2:(i+1)*2]))
    q_d_new_KKT_sol_cart.append(np.array(step_data).flatten())  
    

q_d_new_KKT_sol_cart = np.array(q_d_new_KKT_sol_cart)



# Saving data

In [ ]:
if not standard_result_polar_KKT_new.success:
    print('did not find a solution in control dependent case, not storing the result')
else:
    storage_dict = dict()
    storage_dict["parameters"] = OCP_parameters_np_version
    storage_dict["q_d_new"] = q_d_KKT_new
    storage_dict["q_d_new_cartesian"] = q_d_new_KKT_sol_cart
    storage_dict["lambda_d_new"] = lam_d_KKT_new
    storage_dict["p_y_d_new"] = np.array(p_y_d_new,dtype=float)
    storage_dict["U1_d_new"] = np.array(U1_d_KKT_new,dtype=float)
    storage_dict["U2_d_new"] = np.array(U2_d_KKT_new,dtype=float)
    storage_dict["u_d_new"] = np.array(u_vec_KKT_new,dtype=float)
    storage_dict["I_d_new"] = np.array(I_new_evo,dtype=float)
    storage_dict["H_control_new"] = np.array(H_control_KKT_new,dtype=float)
    storage_dict["v_y_d_new"] = np.array(v_y_d_new,dtype=float)
    storage_dict["mu_sol"] = mu_KKT_new
    storage_dict["nu_sol"] = nu_KKT_new
    storage_dict['dt_new'] = dt_new
    storage_dict['Dt_new'] = Dt_new
    storage_dict['nfev_new'] = standard_result_polar_KKT_new.nfev
    storage_dict['dt_new_no_u'] = dt_new_no_u
    storage_dict['Dt_new_no_u'] = Dt_new_no_u
    storage_dict['nfev_new_no_u'] = standard_result_polar_KKT_new_no_u.nfev



    dirpath = "data/" + OCP_parameters_np_version["folder_name"]+"/data_"+"a=" + str(OCP_parameters_np_version["alpha"])  +"g=" + str(OCP_parameters_np_version["gamma"])
    Path(dirpath).mkdir(parents=True, exist_ok=True)

    file_name = dirpath+"/data_" +"a=" + str(OCP_parameters_np_version["alpha"])  +"g=" + str(OCP_parameters_np_version["gamma"])  +"N=" + str(OCP_parameters_np_version["N"]) + ".pkl"
    with open(file_name, 'wb') as ffile:
        pickle.dump(storage_dict, ffile)
        ffile.close()